# Sweep Analysis: flip_angle

**BlochSimulator Version**: 1.1.0

**Sweep Parameter**: flip_angle

**Data file**: `test_sweep_data.npz`

**Mode**: Dynamic (Time-Resolved)

## Installation

If you haven't installed the `blochsimulator` package yet, you can do so using pip:

```bash
# From GitHub (latest version)
!pip install git+https://github.com/LucaNagel/bloch_sim_gui.git

# From local directory (if you have the source code)
# !pip install .
```

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import xarray as xr
from pathlib import Path

# Set matplotlib style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## Load Sweep Data

In [ ]:
filename = 'test_sweep_data.npz'
is_dynamic = True
file_path = Path(filename)

constant_params = {}
time_vector = None

if file_path.suffix == '.npz':
    data = np.load(file_path, allow_pickle=True)
    param_values = data['parameter_values']
    param_name = str(data['parameter_name'])
    # Load constant params
    if 'constant_params' in data:
        try:
            val = data['constant_params']
            if hasattr(val, 'item'): val = val.item()
            constant_params = json.loads(str(val))
        except:
            pass
    if 'time' in data:
        time_vector = data['time']
    # Load metrics into a dictionary
    results = {k: data[k] for k in data.files if k not in ['parameter_values', 'parameter_name', 'constant_params', 'time']}
elif file_path.suffix == '.csv':
    # Load CSV using numpy (ignoring header row)
    with open(file_path, 'r') as f:
        header_lines = []
        pos = f.tell()
        line = f.readline()
        while line.startswith('#'):
            header_lines.append(line)
            pos = f.tell()
            line = f.readline()
        f.seek(pos) # Go back to first data line
        col_header = line.strip().split(',')
    
    # Parse constant params from header
    for line in header_lines:
        if 'Constant Parameters:' in line:
            try:
                json_str = line.split('Constant Parameters:', 1)[1].strip()
                constant_params = json.loads(json_str)
            except:
                pass
    
    raw_data = np.genfromtxt(file_path, delimiter=',', comments='#', skip_header=1)
    # If only one line, genfromtxt returns 1D array
    if raw_data.ndim == 1:
        raw_data = raw_data.reshape(1, -1)
    
    param_name = col_header[0]
    param_values = raw_data[:, 0]
    
    results = {}
    for i, col_name in enumerate(col_header[1:]):
        results[col_name] = raw_data[:, i+1]
        
    # Check for array sidecar
    array_path = file_path.with_name(file_path.stem + '_arrays.npz')
    if array_path.exists():
        print(f'Loading array data from {array_path.name}')
        arrays = np.load(array_path, allow_pickle=True)
        if 'time' in arrays:
             time_vector = arrays['time']
        # Load constant params from sidecar if not in CSV header
        if not constant_params and 'constant_params' in arrays:
            try:
                val = arrays['constant_params']
                if hasattr(val, 'item'): val = val.item()
                constant_params = json.loads(str(val))
            except: pass
        for k in arrays.files:
            if k not in ['parameter_name', 'parameter_values', 'constant_params', 'time']:
                results[k] = arrays[k]
else:
    raise ValueError('Unsupported file format')

print(f'Loaded sweep data for parameter: {param_name}')
print(f'Steps: {len(param_values)}')
print(f'Metrics: {list(results.keys())}')

## Xarray Dataset Construction

In [ ]:
# Create xarray Dataset from sweep results
data_vars = {}
coords = {param_name: param_values}

if time_vector is not None:
    coords['time'] = time_vector

# Extract spatial/frequency info from constant params
n_pos = constant_params.get('num_positions', 1)
n_freq = constant_params.get('num_frequencies', 1)
n_time = len(time_vector) if time_vector is not None else 0

for k, v in results.items():
    if np.ndim(v) == 1 and len(v) == len(param_values):
        # Scalar metric vs parameter
        data_vars[k] = ([param_name], v)
    elif np.ndim(v) > 1 and len(v) == len(param_values):
        # Dynamic/Multi-dim metric: (param_steps, ...)
        dims = [param_name]
        remaining_shape = v.shape[1:]

        # Try to intelligently name dimensions
        for i, dim_len in enumerate(remaining_shape):
            if n_time > 0 and dim_len == n_time:
                dims.append('time')
            elif n_pos > 1 and dim_len == n_pos:
                dims.append('position')
            elif n_freq > 1 and dim_len == n_freq:
                dims.append('frequency')
            else:
                dims.append(f'dim_{i+1}')

        # Handle duplicate dimension names (if any)
        seen = {}
        for i, d in enumerate(dims):
            if d in seen:
                seen[d] += 1
                dims[i] = f"{d}_{seen[d]}"
            else:
                seen[d] = 0

        data_vars[k] = (dims, v)

ds = xr.Dataset(
    data_vars,
    coords=coords
)
# Add constant params as attrs
if constant_params:
    ds.attrs.update(constant_params)

print('Xarray Dataset created:')
print(ds)

## Simulation Configuration

In [ ]:
print(f'Sweep Mode: {"Dynamic (Time-Resolved)" if is_dynamic else "Static (Final State)"}')
print('\nConstant Parameters (Fixed during sweep):')

# Organize parameters for display if possible
categories = {'Tissue': [], 'Sequence': [], 'Simulation': [], 'Other': []}

if constant_params:
    for k, v in sorted(constant_params.items()):
        if k in ['t1', 't2', 't2_star', 'density', 'name', 'tissue_name']:
            categories['Tissue'].append((k, v))
        elif k in ['te', 'tr', 'flip_angle', 'sequence_type']:
            categories['Sequence'].append((k, v))
        elif k in ['num_positions', 'num_frequencies', 'time_step_us']:
            categories['Simulation'].append((k, v))
        else:
            categories['Other'].append((k, v))

    for cat, items in categories.items():
        if items:
            print(f'\n{cat}:')
            for k, v in items:
                print(f'  {k}: {v}')
else:
    print('  No constant parameters found in metadata.')

if time_vector is not None:
    print(f'\nTime vector loaded: {len(time_vector)} points, duration={time_vector[-1]*1000:.1f} ms')

# Example: Extracting specific parameters for further calculation
t1_ms = constant_params.get('t1', 0) * 1000
te_ms = constant_params.get('te', 0) * 1000
print(f'\nSelected T1: {t1_ms:.1f} ms, TE: {te_ms:.1f} ms')

## Scalar Metrics vs Parameter

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Plot all scalar metrics using xarray
has_scalar = False
for var_name in ds.data_vars:
    if ds[var_name].ndim == 1:
        has_scalar = True
        ds[var_name].plot(ax=ax, marker='o', label=var_name)

if has_scalar:
    ax.set_title(f'Sweep Results: {param_name}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()
else:
    print('No scalar metrics found to plot.')
    plt.close()

## Dynamic Data Analysis

Analysis of time-resolved signals across the parameter sweep.

In [ ]:
# 1. Heatmap of the signal magnitude
dynamic_vars = [v for v in ds.data_vars if ds[v].ndim > 1]
if dynamic_vars:
    target = 'Signal' if 'Signal' in dynamic_vars else dynamic_vars[0]
    print(f'Plotting heatmap for: {target}')

    plt.figure(figsize=(12, 6))
    plot_data = np.abs(ds[target])

    # Reduce dimensions until 2D (sweep_dim, time_dim)
    while plot_data.ndim > 2:
        # Average over intermediate dims (e.g. spatial)
        plot_data = plot_data.mean(dim=plot_data.dims[1])

    plot_data.plot(cmap='viridis')
    plt.title(f'{target} Heatmap')
    plt.show()

In [ ]:
# 2. Coordinate Selection Plot (Data vs Time)
# Demonstrates xarray's powerful selection capabilities
if dynamic_vars and 'time' in ds.coords:
    target = 'Signal' if 'Signal' in dynamic_vars else dynamic_vars[0]

    # Select 3 evenly spaced points from the sweep parameter
    param_vals = ds[param_name].values
    indices = np.linspace(0, len(param_vals)-1, 3, dtype=int)
    selected_vals = param_vals[indices]

    plt.figure(figsize=(10, 6))

    for val in selected_vals:
        # Use .sel() to select data by coordinate value
        trace = np.abs(ds[target].sel({param_name: val}, method='nearest'))
        # Handle extra dims if any
        if trace.ndim > 1:
            trace = trace.mean(axis=tuple(range(trace.ndim-1)))

        trace.plot(label=f'{param_name}={val:.2f}')

    plt.title(f'{target} Evolution for selected {param_name}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print('Skipping coordinate plot (requires time dimension)')